In [5]:
from dotenv import load_dotenv
load_dotenv()

MODEL = "anthropic/claude-haiku-5-5"
LOG_DIR = "../logs/walkthrough"
LIMIT = 5

In [6]:
from inspect_ai.model import (
    ChatMessageAssistant,
    ChatMessageSystem,
    ChatMessageUser,
    GenerateConfig,
    get_model
)

In [11]:
model = get_model(MODEL)
out = await model.generate("What is the capital of Australia? One word.")

In [13]:
print(out.completion)
print(out.choices[0].message)
print(out.stop_reason)
print(out.usage)

Canberra
id='Ud2sNUmZN6GUzdCp2p7rbK' content=[ContentText(internal=None, type='text', text='Canberra', refusal=None, citations=None, cache_breakpoint=None)] source='generate' metadata=None role='assistant' tool_calls=None model='claude-haiku-5-5'
stop
input_tokens=21 output_tokens=6 total_tokens=27 input_tokens_cache_write=0 input_tokens_cache_read=0 reasoning_tokens=None total_cost=None


In [16]:
messages = [
    ChatMessageSystem(content="You are a pirate. Answer in one short sentence."),
    ChatMessageUser(content="What is 2 + 2?"),
    ChatMessageAssistant(content="Arr, that be 4, matey."),
    ChatMessageUser(content="And 3 + 3?"),
]
out = await model.generate(messages)
print(out.completion)

Arr, that be 6, matey!


In [22]:
for effort in ["low", "medium", "high", "xhigh"]:
    out = await model.generate(
        "Write a haiku about evals.", config=GenerateConfig(effort=effort)
    )
    print(f"### Effort: {effort}")
    print(out.completion)
    print()

### Effort: low
Benchmarks climb high,
yet the model stumbles on tasks
the test never asked.

### Effort: medium
Scores climb, tests pass clean,
yet the model's quiet flaws
hide beyond the set.

### Effort: high
Ninety-nine percent,
the test was in the training.
Who checks the checker?

### Effort: xhigh
All evals pass green:
the model studied the test,
the real world unseen.



In [25]:
for _ in range(20):
    out = await model.generate(
        "Tell me a funny joke.", config=GenerateConfig(effort="high")
    )

    print(out.completion)
    print("------------")

Why don't scientists trust atoms?

Because they make up everything!
------------
Why don't scientists trust atoms?

Because they make up everything.
------------
Why don't scientists trust atoms?

Because they make up everything!
------------
Why did the scarecrow win an award?

Because he was outstanding in his field!
------------
Why did the scarecrow win an award?

Because he was outstanding in his field!
------------
Why don't scientists trust atoms?

Because they make up everything!
------------
Why don't scientists trust atoms?

Because they make up everything!
------------
Why don't scientists trust atoms?

Because they make up everything!
------------
Why don't skeletons fight each other?

They don't have the guts.
------------
Why don't scientists trust atoms?

Because they make up everything!
------------
Why don't skeletons fight each other?

They don't have the guts.
------------
Why don't scientists trust atoms?

Because they make up everything!
------------
Why don't scie

In [26]:
for i in range(20):
    periods = "."*i
    prompt = f"{periods}Tell me a funny joke."
    out = await model.generate(
        prompt, config=GenerateConfig(effort="high")
    )
    print(prompt)
    print("##")
    print(out.completion)
    print("\n------------\n")

Tell me a funny joke.
##
Why don't scientists trust atoms?

Because they make up everything!

------------

.Tell me a funny joke.
##
Why don't scientists trust atoms?

Because they make up everything!

------------

..Tell me a funny joke.
##
Why don't skeletons fight each other?

They don't have the guts.

------------

...Tell me a funny joke.
##
Why did the scarecrow win an award?

Because he was outstanding in his field.

------------

....Tell me a funny joke.
##
Why don't scientists trust atoms?

Because they make up everything.

------------

.....Tell me a funny joke.
##
Why don't scientists trust atoms?

Because they make up everything!

------------

......Tell me a funny joke.
##
Why don't scientists trust atoms?

Because they make up everything.

------------

.......Tell me a funny joke.
##
Why don't skeletons fight each other?

They don't have the guts.

------------

........Tell me a funny joke.
##
Why don't skeletons fight each other?

They don't have the guts.

-----

# Datasets and Sample

In [27]:
from inspect_ai.dataset import (
    FieldSpec,
    MemoryDataset, 
    Sample,
    example_dataset,
    hf_dataset
)

In [28]:
qa_dataset = MemoryDataset(
    [
        Sample(id="cap-au", input="What is the capital of Australia?", target="Canberra"),
        Sample(id="cap-ca", input="What is the capital of Canada?", target="Ottawa"),
        Sample(id="cap-ch", input="What is the capital of Switzerland?", target="Bern"),
        Sample(id="math-1", input="What is 17 * 23?", target="391", metadata={"kind": "math"}),
        Sample(id="math-2", input="What is 2 ** 10?", target="1024", metadata={"kind": "math"}),
    ],
    name="tiny_qa",
)

In [29]:
print(len(qa_dataset), qa_dataset[0])

5 input='What is the capital of Australia?' choices=None target='Canberra' id='cap-au' description=None metadata=None sandbox=None files=None setup=None checkpoint=None


In [31]:
chat_sample = Sample(
    input = [
        ChatMessageSystem(content="Answer with a single word."),
        ChatMessageUser(content="Opposite of hot?"),
    ],
    target="cold"
)

In [34]:
mc_dataset = MemoryDataset(
    [
        Sample(
            input="Which planet is closest to the sun?",
            choices=["Venus", "Mercury", "Mars", "Earth"],
            target="B"
        ),
        Sample(
            input="Which gas do plants absorb for photosynthesis?",
            choices=["Oxygen", "Nitrogen", "Carbon dioxide", "Helium"],
            target="C",
        )
    ]
)

In [35]:
security = example_dataset("security_guide")
print(security[0])

input=[ChatMessageUser(id='AkTbFnPngK7eNfZNiDT4bp', content='How do I prevent SQL Injection attacks?', source='input', metadata=None, role='user', tool_call_id=None)] choices=None target='use parameterized queries and prepared statements' id=None description=None metadata=None sandbox=None files=None setup=None checkpoint=None


In [36]:
sample = security[0]
sample.input

[ChatMessageUser(id='AkTbFnPngK7eNfZNiDT4bp', content='How do I prevent SQL Injection attacks?', source='input', metadata=None, role='user', tool_call_id=None)]

In [37]:
sample.target

'use parameterized queries and prepared statements'

In [39]:
gsm8k_fieldspec = hf_dataset(
    "openai/gsm8k",
    name="main",
    split="test",
    sample_fields=FieldSpec(input="question", target="answer"),
    limit=20
)

In [40]:
gsm8k_fieldspec[0]

Sample(input="Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?", choices=None, target='Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.\nShe makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.\n#### 18', id=None, description=None, metadata=None, sandbox=None, files=None, setup=None, checkpoint=None)

In [41]:
def gsm8k_record_to_sample(record: dict) -> Sample:
    return Sample(
        input=record['question'],
        target=record['answer'].split("####")[-1].strip(),
        metadata={"full_solution": record["answer"]}
    )

gsm8k = hf_dataset(
    "openai/gsm8k", name="main", split="test", sample_fields=gsm8k_record_to_sample, limit=20
)

print(gsm8k_fieldspec[0].target[-40:], "|", gsm8k[0].target)

very day at the farmer’s market.
#### 18 | 18


In [48]:
qa_dataset.shuffle(seed=0)
math_only = qa_dataset.filter(lambda s: (s.metadata or {}).get("kind") == "math")
print(len(math_only), [s.id for s in math_only])
print([s.id for s in qa_dataset[:2]])

2 ['math-2', 'math-1']
['cap-ch', 'cap-ca']


# Smallest possible eval

In [49]:
from inspect_ai import Task, eval, task
from inspect_ai.scorer import includes, match
from inspect_ai.solver import generate

In [50]:
logs = eval(
    Task(dataset=qa_dataset, solver=generate(), scorer=includes()),
    model=MODEL,
    log_dir=LOG_DIR
)

Output()

In [56]:
log = logs[0]

print(log.status)
print(log.location)
print(log.results.scores[0].name, log.results.scores[0].metrics)
print(log.stats.model_usage)

success
/Users/traviswhitfield/Documents/github/my_projects/trav_ai_control_sandbox/logs/walkthrough/2026-11-12T20-10-14-00-00_task_A3UzhSWzoBJvSPRvReVXN8.eval
includes {'accuracy': EvalMetric(name='accuracy', group=None, value=1.0, params={}, metadata=None), 'stderr': EvalMetric(name='stderr', group=None, value=0.0, params={}, metadata=None)}
{'anthropic/claude-haiku-5-5': ModelUsage(input_tokens=91, output_tokens=224, total_tokens=315, input_tokens_cache_write=0, input_tokens_cache_read=0, reasoning_tokens=None, total_cost=None)}


In [57]:
s = log.samples[0]
print(s.id, "|", s.output.completion[:80], "|", s.scores)

cap-au | The capital of Australia is **Canberra**. Although Sydney and Melbourne are larg | {'includes': Score(value='C', answer='the capital of australia is **canberra**. although sydney and melbourne are larger and better known, canberra was purpose-built as the capital, chosen as a compromise between those two rival cities.', explanation='The capital of Australia is **Canberra**. Although Sydney and Melbourne are larger and better known, Canberra was purpose-built as the capital, chosen as a compromise between those two rival cities.', reason=None, metadata=None, history=[])}


In [58]:
for m in s.messages:
    print(type(m).__name__, ":", m.text[:80])

ChatMessageUser : What is the capital of Australia?
ChatMessageAssistant : The capital of Australia is **Canberra**. Although Sydney and Melbourne are larg


In [60]:
@task
def tiny_qa(scorer_name: str = "includes"):
    return Task(
        dataset=qa_dataset,
        solver=generate(),
        scorer=includes() if scorer_name == "includes" else match()
    )

In [62]:
logs = eval(tiny_qa(scorer_name="match"), model=MODEL, log_dir=LOG_DIR)

Output()

In [65]:
logs = eval(tiny_qa(scorer_name="includes"), model=MODEL, log_dir=LOG_DIR)

Output()

# Solvers and TaskState

In [66]:
from inspect_ai.solver import (
    Generate,
    TaskState,
    chain,
    chain_of_thought,
    multiple_choice,
    prompt_template,
    self_critique,
    solver,
    system_message
)

In [67]:
solver_variants = {
    'plain': [generate()],
    'system_message': [system_message("Answer in as few words as possible."), generate()],
    "prompt_template": [prompt_template("Question: {prompt}\nGive only the answer."), generate()],
    "chain_of_thought": [chain_of_thought(), generate()],
    "self_critique": [generate(), self_critique()],
}

for name, solvers in solver_variants.items():
    eval(
        Task(
            dataset=qa_dataset,
            solver=solvers, 
            scorer=includes(),
            name=f"solver_{name}"
        ),
        model=MODEL,
        log_dir=LOG_DIR
    )

Output()

Output()

Output()

Output()

Output()

In [68]:
from inspect_ai.scorer import choice

eval(Task(dataset=mc_dataset, solver=multiple_choice(), scorer=choice()), model=MODEL, log_dir=LOG_DIR)

Output()

In [70]:
concise_cot = chain(system_message("Be concise"), chain_of_thought(), generate())

eval(
    Task(
        dataset=qa_dataset,
        solver=concise_cot, 
        scorer=includes(),
    ),
    model=MODEL,
    log_dir=LOG_DIR
)

Output()

In [72]:
@solver
def add_hint(hint: str = "Think carefully before answering."):
    async def solve(state: TaskState, generate: Generate) -> TaskState:
        state.user_prompt.text = f"{state.user_prompt.text}\n\n{hint}"
        return await generate(state)

    return solve

eval(Task(dataset=qa_dataset, solver=add_hint(), scorer=includes()), model=MODEL, log_dir=LOG_DIR)

Output()

In [73]:
@solver
def best_of_two():
    async def solve(state: TaskState, generate: Generate):
        model = get_model()
        a = await model.generate(state.messages)
        b = await model.generate(state.messages)
        chosen = a if len(a.completion) <= len(b.completion) else b
        state.messages.append(chosen.message)
        state.output = chosen
        state.metadata['candidates'] = [a.completion, b.completion]
        return state
    return solve

eval(Task(dataset=qa_dataset, solver=best_of_two(), scorer=includes()), model=MODEL, log_dir=LOG_DIR)

Output()

In [74]:
@solver
def critic(critic_model: str = MODEL):
    async def solve(state: TaskState, generate: Generate):
        state = await generate(state)
        review = await get_model(critic_model).generate(
            f"Question: {state.input_text}\nAnswer: {state.output.completion}\n"
            "Is this answer correct? Reply in one sentence."
        )
        state.messages.append(ChatMessageUser(content=f"A reviewer said: {review.completion}\nGive your final answer."))
        return await generate(state)
    return solve

eval(Task(dataset=qa_dataset, solver=critic(), scorer=includes()), model=MODEL, log_dir=LOG_DIR)

Output()

In [77]:
qa_dataset.samples

[Sample(input='What is the capital of Switzerland?', choices=None, target='Bern', id='cap-ch', description=None, metadata=None, sandbox=None, files=None, setup=None, checkpoint=None),
 Sample(input='What is the capital of Canada?', choices=None, target='Ottawa', id='cap-ca', description=None, metadata=None, sandbox=None, files=None, setup=None, checkpoint=None),
 Sample(input='What is the capital of Australia?', choices=None, target='Canberra', id='cap-au', description=None, metadata=None, sandbox=None, files=None, setup=None, checkpoint=None),
 Sample(input='What is 2 ** 10?', choices=None, target='1024', id='math-2', description=None, metadata={'kind': 'math'}, sandbox=None, files=None, setup=None, checkpoint=None),
 Sample(input='What is 17 * 23?', choices=None, target='391', id='math-1', description=None, metadata={'kind': 'math'}, sandbox=None, files=None, setup=None, checkpoint=None)]

In [ ]:
@solver
def math():
    async def solve(state: TaskState, generate: Generate):
        if state.metadata.get("kind") == "math":
            state.completed = True
        return await generate(state)
    return solve

eval(Task(dataset=qa_dataset, solver=math(), scorer=includes()), model=MODEL, log_dir=LOG_DIR)

Output()

In [79]:
@solver
def skip_math():
    async def solve(state: TaskState, generate: Generate) -> TaskState:
        if state.metadata.get("kind") == "math":
            state.completed = True  # the plan loop checks this after this solver returns
        return state  # no generate() here
    return solve
            
eval(
    Task(dataset=qa_dataset, solver=[skip_math(), generate()], scorer=includes()),
    model=MODEL,
    log_dir=LOG_DIR,
)

Output()

# Scorers and metrics

In [80]:
from inspect_ai.scorer import (
    CORRECT,
    INCORRECT,
    SampleScore,
    Score,
    Target,
    accuracy,
    exact,
    f1,
    mean,
    metric,
    model_graded_fact,
    pattern,
    scorer,
    stderr,
)

In [83]:
logs = eval(
    Task(
        dataset=qa_dataset,
        solver = [system_message("Answer in a full sentence."), generate()],
        scorer = [includes(), match(), exact(), f1(), model_graded_fact()]
    ),
    model=MODEL,
    log_dir=LOG_DIR
)

for r in logs[0].results.scores:
    print(r.name, {k: round(v.value, 3) for k, v in r.metrics.items()})

Output()

includes {'accuracy': 1.0, 'stderr': 0.0}
match {'accuracy': 0.8, 'stderr': 0.2}
exact {'mean': 0.0, 'stderr': 0.0}
f1 {'mean': 0.254, 'stderr': 0.062}
model_graded_fact {'accuracy': 1.0, 'stderr': 0.0}


In [84]:
eval(
    Task(
        dataset=qa_dataset,
        solver=[chain_of_thought(), generate()],
        scorer=pattern(r"ANSWER:\s*(.+)"),
    ),
    model=MODEL,
    log_dir=LOG_DIR
)

Output()

In [86]:
@metric
def fraction_perfect():
    def compute(scores: list[SampleScore]):
        return sum(s.score.as_float() == 1.0 for s in scores) / len(scores)
    return compute

In [90]:
# Custom metric: receives every SampleScore, returns one number.
@metric
def fraction_perfect():
    def compute(scores: list[SampleScore]) -> float:
        return sum(s.score.as_float() == 1.0 for s in scores) / len(scores)

    return compute


# Custom scorer #1: CORRECT/INCORRECT plus an explanation (shown in the viewer).
@scorer(metrics=[accuracy(), stderr(), fraction_perfect()])
def short_and_right(max_words: int = 5):
    async def score(state: TaskState, target: Target) -> Score:
        answer = state.output.completion
        right = target.text.lower() in answer.lower()
        short = len(answer.split()) <= max_words
        return Score(
            value=CORRECT if (right and short) else INCORRECT,
            answer=answer,
            explanation=f"right={right}, words={len(answer.split())}",
        )

    return score

In [88]:
# Custom scorer #2: a dict value -> each key gets its own metrics.
@scorer(metrics={"correct": [accuracy()], "length": [mean()]})
def correct_and_length():
    async def score(state: TaskState, target: Target) -> Score:
        answer = state.output.completion
        return Score(
            value={
                "correct": (
                    CORRECT if target.text.lower() in answer.lower() else INCORRECT
                ),
                "length": len(answer.split()),
            },
            answer=answer,
        )

    return score

In [91]:
logs = eval(
    Task(
        dataset=qa_dataset,
        solver=generate(),
        scorer=[short_and_right(), correct_and_length()],
        # Task(metrics=[...]) would override the metrics of *every* scorer here, which
        # would break on correct_and_length's dict value. So attach metrics per scorer.
    ),
    model=MODEL,
    log_dir=LOG_DIR,
)
for r in logs[0].results.scores:
    print(r.name, {k: round(v.value, 3) for k, v in r.metrics.items()})

Output()

short_and_right {'accuracy': 0.2, 'stderr': 0.2, 'fraction_perfect': 0.2}
correct {'accuracy': 1.0}
length {'mean': 19.6}


# Tools

In [92]:
from inspect_ai.solver import use_tools
from inspect_ai.tool import bash, python, tool

In [93]:
@tool
def multiply():
    async def execute(x: float, y:float):
        """Multiply two numbers.

        Args:
            x: First number.
            y: Second number.

        Returns:
            The product x * y.
        """
        return x * y

    return execute

In [94]:
arith = MemoryDataset(
    [
        Sample(input="Use the tool to compute 123.5 * 47.", target="5804.5"),
        Sample(input="Use the tool to compute 991 * 1013.", target="1003883"),
    ]
)

In [95]:
eval(
    Task(dataset=arith, solver=[use_tools(multiply()), generate()], scorer=includes()),
    model=MODEL,
    log_dir=LOG_DIR
)

Output()

In [96]:
eval(
    Task(dataset=arith, solver=[use_tools(multiply(), tool_choice=None), generate()], scorer=includes()),
    model=MODEL,
    log_dir=LOG_DIR
)

Output()

In [97]:
from inspect_ai.util import sandbox

In [98]:
@tool
def count_lines():
    async def execute(path: str):
        """Count lines in a file.

        Args:
            path: Path to the file.
        """
        result = await sandbox().exec(
            ['wc', '-l', path]
        ) 
        return result.stdout if result.success else result.stderr

    return execute

In [100]:
eval(
    Task(
        dataset=[
            Sample(input="How many lines are in ../pyproject.toml? Use a tool.", target="")
        ],
        solver=[use_tools(count_lines(), python()), generate()],
        sandbox='local',
    ),
    model=MODEL,
    log_dir=LOG_DIR
)

Output()

In [101]:
from pathlib import Path

eval(
    Task(
        dataset=[
            Sample(
                input="How many lines are in pyproject.toml? Use a tool.",
                target="",
                # key = path inside the sandbox, value = source file on your machine (or inline text)
                files={"pyproject.toml": str(Path("../pyproject.toml").resolve())},
            )  
        ],
        solver=[use_tools(count_lines(), python()), generate()],
        sandbox="local",
    ),  
    model=MODEL,
    log_dir=LOG_DIR,
)

Output()

# Agents

In [102]:
from inspect_ai.agent import AgentState, agent, as_tool, handoff, react
from inspect_ai.model import execute_tools

In [103]:
eval(
    Task(
        dataset=arith,
        solver = react(
            prompt="Solve the problem, then submit the number.",
            tools=[multiply()],
            attempts=2
        ),
        scorer=includes(),
        message_limit=20
    ),
    model=MODEL,
    log_dir=LOG_DIR
)

Output()

In [104]:
@agent
def tool_loop(tools, max_turns: int=5):
    async def execute(state: AgentState):
        model=get_model()
        for _ in range(max_turns):
            state.output = await model.generate(state.messages, tools=tools)
            state.messages.append(state.output.message)
            if not state.output.message.tool_calls:
                break
            result = await execute_tools(state.messages, tools)
            state.messages.extend(result.messages)
        return state

    return execute

In [105]:
eval(
    Task(dataset=arith, solver=tool_loop(tools=[multiply()]), scorer=includes()),
    model=MODEL,
    log_dir=LOG_DIR,
)

Output()

In [106]:
math_agent = react(
    name="math_agent",
    description="Expert at arithmetic. Give it a calculation.",
    tools=[multiply()],
)
supervisor = react(
    prompt="Delegate arithmetic to the math agent.", tools=[handoff(math_agent)]
)

eval(
    Task(dataset=arith, solver=supervisor, scorer=includes()),
    model=MODEL,
    log_dir=LOG_DIR,
)

Output()

[11/12/26 19:16:11] WARNING  sample=2                                                                 ]8;id=15626563;file:///Users/traviswhitfield/Documents/github/my_projects/trav_ai_control_sandbox/.venv/lib/python3.13/site-packages/inspect_ai/_util/logger.py\logger.py]8;;\:]8;id=15626564;file:///Users/traviswhitfield/Documents/github/my_projects/trav_ai_control_sandbox/.venv/lib/python3.13/site-packages/inspect_ai/_util/logger.py#279\279]8;;\
                             anthropic model 'claude-haiku-5-5' dropped replayed thinking block(s)                 
                             from the request (reason: prefix_binding_mismatch), so their reasoning                
                             is no longer visible to the model. A prefix_binding_mismatch reason                   
                             means earlier conversation content (system prompt, tools, or messages)                
                             changed after the blocks were produced. Per-request details are recorded              
                             in the model output metadata under extra_body.input_transformations.                  

In [107]:
math_agent = react(
    name="math_agent",
    description="Expert at arithmetic. Give it a calculation.",
    tools=[multiply()],
)
supervisor = react(
    prompt="Delegate arithmetic to the math agent.", tools=[as_tool(math_agent)]
)

eval(
    Task(dataset=arith, solver=supervisor, scorer=includes()),
    model=MODEL,
    log_dir=LOG_DIR,
)

Output()

# State across steps: metadata vs store

In [108]:
from inspect_ai.util import StoreModel, store_as
from pydantic import Field

In [109]:
class Attempts(StoreModel):
    drafts: list[str] = Field(default_factory=list)

In [110]:
@solver
def draft_twice():
    async def solve(state: TaskState, generate: Generate):
        attempts = store_as(Attempts)
        for _ in range(2):
            out = await get_model().generate(state.messages)
            attempts.drafts.append(out.completion)
        return await generate(state)

    return solve

In [111]:
@scorer(metrics=[mean()])
def drafts_agree():
    async def score(state: TaskState, target: Target) -> Score:
        drafts = state.store_as(Attempts).drafts  # read what the solver wrote
        return Score(value=float(len(set(drafts)) == 1), explanation=str(drafts))

    return score


eval(
    Task(dataset=qa_dataset, solver=draft_twice(), scorer=[includes(), drafts_agree()]),
    model=MODEL,
    log_dir=LOG_DIR,
    temperature=1.0,
)

Output()

# Running many evals

In [112]:
logs = eval(
    [tiny_qa(), tiny_qa(scorer_name="match")],
    model = [MODEL, "anthropic/claude-sonnet-5-5"],
    log_dir=LOG_DIR
)
print(len(logs))

Output()

4


# Log analysis in Python

In [114]:
from inspect_ai.log import list_eval_logs, read_eval_log, read_eval_log_samples

log_files = list_eval_logs(LOG_DIR)
print(len(log_files))

# header_only=True skips samples: fast, even for huge logs (e.g. your SWE-bench runs)
header = read_eval_log(log_files[0], header_only=True)
print(
    header.eval.task,
    header.eval.model,
    header.results.scores[0].metrics if header.results else None,
)

# Stream samples one at a time instead of loading them all into memory
for sample in read_eval_log_samples(log_files[0]):
    print(sample.id, sample.scores, len(sample.events), "events")



34
tiny_qa anthropic/claude-sonnet-5-5 {'accuracy': EvalMetric(name='accuracy', group=None, value=0.2, params={}, metadata=None), 'stderr': EvalMetric(name='stderr', group=None, value=0.2, params={}, metadata=None)}
cap-ch {'match': Score(value='I', answer='The capital of Switzerland is **Bern**. Officially, Switzerland has no "capital" in law. Bern is designated as the "federal city" (*Bundesstadt*), and it serves as the seat of the Swiss federal government and parliament. Zürich and Geneva are larger or more internationally known, but they aren\'t the capital.', explanation='The capital of Switzerland is **Bern**. Officially, Switzerland has no "capital" in law. Bern is designated as the "federal city" (*Bundesstadt*), and it serves as the seat of the Swiss federal government and parliament. Zürich and Geneva are larger or more internationally known, but they aren\'t the capital.', reason=None, metadata=None, history=[])} 14 events
cap-ca {'match': Score(value='I', answer='The capita

In [116]:
# Dataframes: one row per eval / sample / message / event.
from inspect_ai.analysis import evals_df, messages_df, samples_df

evals = evals_df(LOG_DIR)
print(evals[["task_name", "model", "score_headline_value"]])

samples = samples_df(LOG_DIR)
print(samples.columns.tolist())

# Your earlier MMLU runs, compared across models:
mmlu = evals_df("../logs")
mmlu = mmlu[mmlu.task_name.str.contains("mmlu")]
print(mmlu[["model", "score_headline_value", "total_samples"]])

# EXERCISE: with samples_df, find the samples where includes() and match() disagree
# in the multi-scorer log from section 5.

                  task_name                        model  score_headline_value
0                      task   anthropic/claude-haiku-5-5                   1.0
1                      task   anthropic/claude-haiku-5-5                   0.0
2                      task   anthropic/claude-haiku-5-5                   0.5
3                      task   anthropic/claude-haiku-5-5                  <NA>
4                      task   anthropic/claude-haiku-5-5                   1.0
5                      task   anthropic/claude-haiku-5-5                   1.0
6                      task   anthropic/claude-haiku-5-5                  <NA>
7                      task   anthropic/claude-haiku-5-5                   0.6
8                      task   anthropic/claude-haiku-5-5                   1.0
9                      task   anthropic/claude-haiku-5-5                   1.0
10    solver_system_message   anthropic/claude-haiku-5-5                   1.0
11                  tiny_qa  anthropic/claude-sonnet

In [117]:
mmlu

,eval_id,eval_set_id,run_id,task_id,log,created,tags,git_origin,git_commit,packages,...,score_model_graded_fact_accuracy,score_model_graded_fact_stderr,score_pattern_accuracy,score_pattern_stderr,score_refusal_flag_mean,score_short_and_right_accuracy,score_short_and_right_fraction_perfect,score_short_and_right_stderr,score_swe_bench_scorer_mean,score_swe_bench_scorer_std
0,DDyLDejgDio239HGedVUsT,<NA>,LktkAg384RYok3HkGPjx7R,VL2Acm9UsJze84F3fJ45Pf,/Users/traviswhitfield/Documents/github/my_pro...,2026-11-12 02:42:50+00:00,,https://github.com/thwhitfield/trav_ai_control...,4284d77,"{""inspect_ai"": ""0.3.289"", ""inspect_evals"": ""0....",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,i5pLQb7u8oQ5uqj5TEwwUt,<NA>,mCPfFQpBk5uicapUXPwyD9,k2jAcqJKNWMAwT8chZLRLt,/Users/traviswhitfield/Documents/github/my_pro...,2026-11-12 02:45:17+00:00,,https://github.com/thwhitfield/trav_ai_control...,4284d77,"{""inspect_ai"": ""0.3.289"", ""inspect_evals"": ""0....",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
14,go53icVQewkb7w8aG22XPu,<NA>,NCoQC4o88eX9ZfTVFGASJi,757NzSjEpZMKcskHmCx8qf,/Users/traviswhitfield/Documents/github/my_pro...,2026-11-12 02:59:36+00:00,,https://github.com/thwhitfield/trav_ai_control...,4284d77,"{""inspect_ai"": ""0.3.289"", ""inspect_evals"": ""0....",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
15,7UqKXUB56KN9SD9TV5z3Jj,<NA>,6oVNjEprSziT8qB3sPbFBD,9eLyrxnnZ4reLKMtLpH3Ti,/Users/traviswhitfield/Documents/github/my_pro...,2026-11-12 02:59:24+00:00,,https://github.com/thwhitfield/trav_ai_control...,4284d77,"{""inspect_ai"": ""0.3.289"", ""inspect_evals"": ""0....",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
19,hkKg8Gp9vekpWerAJUzGWp,<NA>,hfyDHkvAsRQDt6n9fVD4HU,LhGYhgLxczzCTzxekZMq9X,/Users/traviswhitfield/Documents/github/my_pro...,2026-11-12 03:00:50+00:00,,https://github.com/thwhitfield/trav_ai_control...,4284d77,"{""inspect_ai"": ""0.3.289"", ""inspect_evals"": ""0....",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
22,BpzQkfeX9qhFhpH8mgxnCp,<NA>,Z8qD6Piv7Hzpc49i9CzH2E,gq7kx6HFm9QAcUcvVBfx7Z,/Users/traviswhitfield/Documents/github/my_pro...,2026-11-12 03:01:44+00:00,,https://github.com/thwhitfield/trav_ai_control...,4284d77,"{""inspect_ai"": ""0.3.289"", ""inspect_evals"": ""0....",...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
